In [ ]:
!pip install litellm

In [ ]:
from litellm import completion

In [3]:
import os
os.environ["EDENAI_API_KEY"] = "..."

## Simple Call to Openai Through Edenai

In [2]:
# openai call
response = completion(
    model = "edenai/openai/gpt-4o",
    messages = [{ "content": "Hello, how are you?","role": "user"}],
    set_verbose = True
)
print("Edenai gpt-4o model Response : \n")
print(response)
 

Edenai gpt-4o model Response : 

ModelResponse(id='chatcmpl-1fcf3629-a963-4a6b-b972-a2ad2de1244d', created=1736938508, model=None, object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content="Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", role='assistant', tool_calls=None, function_call=None))], usage=Usage(completion_tokens=31, prompt_tokens=13, total_tokens=44, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None)))


## Multimodal Call to Gemini with EdenAI


In [3]:
response = completion(
    model="edenai/google/gemini-1.5-flash",
    messages=[
    {
        "role": "user",
        "content": """[
            {
                "type": "text",
                "content": {
                    "text": "is there a lizard in the image?"
                }
            },
            {
                "type": "media_url",
                "content": {
                    "media_url": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS53fbiM_1J_a_gPfGctjxQUWRxhj3-l8ueSw&s",
                    "media_type": "image/jpeg"
                }
            }
        ]"""
    }
],
)
print("Edenai gemini multimodal request Response : \n")
print(response)


Edenai gemini multimodal request Response : 

ModelResponse(id='chatcmpl-fc87d082-605c-4681-95fa-0224869701eb', created=1736938510, model=None, object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='Yes, there is a lizard in the image.  It appears to be a type of agama lizard, possibly a species of *Agama*.  The vibrant blue throat patch is a distinctive feature.\n', role='assistant', tool_calls=None, function_call=None))], usage=Usage(completion_tokens=41, prompt_tokens=267, total_tokens=308, completion_tokens_details=None, prompt_tokens_details=None))


# CALL WITH TOOLS


In [4]:
import json

def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": "celsius"})
    elif "san francisco" in location.lower():
        return json.dumps({"location": "San Francisco", "temperature": "72", "unit": "fahrenheit"})
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": "celsius"})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

messages = [{"role": "user", "content": "What's the weather like in San Francisco, Tokyo, and Paris?"}]
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [5]:
response = completion(
    model = "edenai/openai/gpt-4o",
    messages = messages,
    tools = tools
)
print("\nLLM Response1:\n", response)
response_message = response.choices[0].message
tool_calls = response.choices[0].message.tool_calls
print("\nTool Choice:\n", tool_calls)


LLM Response1:
 ModelResponse(id='chatcmpl-84e4bd91-8067-47e5-b38c-5ea3433babad', created=1736938515, model=None, object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(function=Function(arguments='{"location": "San Francisco, CA"}', name='get_current_weather'), id='call_VBbIBCH31YEMgPbQItTh64Bd', type='function'), ChatCompletionMessageToolCall(function=Function(arguments='{"location": "Tokyo, Japan"}', name='get_current_weather'), id='call_3yLMYKMAjLW2PWsVumc52dV9', type='function'), ChatCompletionMessageToolCall(function=Function(arguments='{"location": "Paris, France"}', name='get_current_weather'), id='call_yA2epA1AhblHCziz5VL9h0hB', type='function')], function_call=None))], usage=Usage(completion_tokens=69, prompt_tokens=85, total_tokens=154, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=0, audio_tokens=0, reas